# CS-781 Capstone — Phase 6: Abstractive Legal Summarization

This notebook fine-tunes **your own Phase-1 language models** (A, B, C, D) to write a summary of a legal document, and writes one submission zip per experiment with a **descriptive file name**, e.g. `phase6_S2_B-GQA-RoPE_HoulsbyAdapter_headtail_valRL0.2512.zip`, all collected in `cs781_phase6/submissions/`.

## Before you run it
1. **Settings (right panel):** Accelerator **GPU T4 x2**, Internet **on**.
2. **Add-ons → Secrets:** turn on the secret **`HF`** (your Hugging Face token). The data is downloaded from `Exploration-Lab/CS-781-Capstone/Phase_6_Summ`.
3. **Add Input:** your Phase-1 dataset (the same one as in Phases 2–5). Section 6 prints which file is used for A, B, C and D.
4. *Save Version → Save & Run All.* To continue later, attach this notebook's previous output too: finished experiments, trained models and generated summaries are loaded, not redone.

## The approach
- **Decoder models (A, B, D)** read one sequence `[document] <s> [summary] </s>` and learn to continue a document with its summary (loss on the summary tokens only). **C** uses its encoder for the document and its decoder for the summary; its encoder input embeddings are scaled up, because its Phase-1 encoder ignored the words.
- **Long documents:** the token budget is split into the first 60% and the last 40% of the document (facts at the start, decision at the end). The summary budget comes from the data (90th percentile of the reference length, at most half of the 2,048-token context).
- **No dev split is provided**, so 5% of train.json (at most 300 documents) is held out: the held-out loss picks the best epoch, and 60 held-out documents are summarised to choose the decoding setting and report ROUGE (and BERTScore, if the package is installed).
- **Decoding:** greedy with a key/value cache, repetition penalty and n-gram blocking (the Phase-1 decoding study), a minimum and maximum length from the data.

| Submission | Model | Technique (Phase-6 list) |
|---|---|---|
| S2_B_houlsby | B (GQA/RoPE) | **Houlsby adapters** (after attention and feed-forward) |
| S1_D_lorma | D (GPT, 400M tokens) | **LoRMA** (compulsory), query/value, rank 32 |
| S3_A4_kd | A (first 4 layers) | **sequence-level knowledge distillation** from S2 (the student also learns S2's summaries) |
| S4_C_lora | C (encoder-decoder) | **LoRA** on every linear layer of encoder and decoder |

Extra experiments (Section 2): Pfeiffer vs Houlsby adapters, document start only vs start + end.

## Rules this follows
Every experiment starts from your Phase-1 model; no other pretrained model, weights or API is used for training or generation. One summary per test document, with its original id — checked before every zip is written.

## 1. Setup and Hugging Face login

In [1]:
import os
import re
import sys
import json
import math
import time
import glob
import gc
import pickle
import copy
import random
import shutil
import zipfile
import hashlib
import warnings
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print("Kaggle secret 'HF' not available (" + type(e).__name__ + "); using the HF_TOKEN environment variable if set.")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face login successful.")
    except Exception as e:
        print("Hugging Face login failed:", repr(e))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("PyTorch:", torch.__version__, "| device:", DEVICE)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: no GPU. Use Kaggle 'GPU T4 x2' for real runs.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
PyTorch: 2.11.0+cu128 | device: cuda
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Configuration and experiments

Each experiment is one line: which Phase-1 model, which technique. `RUN_EXPERIMENTS` chooses what runs.

In [2]:
SEED = 42
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE6_DIR = "Phase_6_Summ"

INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
WORK_ROOT = os.environ.get("CS781_P6_WORK_ROOT",
                           "/kaggle/working/cs781_phase6" if os.path.isdir("/kaggle/working") else os.path.abspath("cs781_phase6"))
LOCAL_DATA_DIR = os.environ.get("CS781_P6_DATA_DIR", "")
SUBMISSION_DIR = os.path.join(WORK_ROOT, "submissions")
CACHE_DIR = os.path.join(WORK_ROOT, "cache")
for _d in (WORK_ROOT, SUBMISSION_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

PHASE1_MODELS = {
    "A": dict(architecture="A", pick="other"),
    "B": dict(architecture="B", pick="best"),
    "C": dict(architecture="C", pick="best"),
    "D": dict(architecture="A", pick="best"),
}
MODEL_TAGS = {"A": "A-GPT220M", "B": "B-GQA-RoPE", "C": "C-EncDec", "D": "D-GPT400M"}

VAL_FRACTION, VAL_MAX, VAL_MIN = 0.05, 300, 40
GEN_EVAL_DOCS = 60

DEFAULTS = dict(
    backbone="B", n_layers=None,
    technique="adapter",
    doc_selection="head_tail",
    tokens_per_batch=8192,
    epochs=4, patience=2,
    lr_backbone=3e-5, lr_peft=5e-4, lr_head=1e-3,
    weight_decay=0.01, warmup_frac=0.05, grad_clip=1.0, dropout=0.1,
    lora_r=16, lora_alpha=32, lora_scaling="standard", peft_targets="qv", lorma_variant="plus",
    adapter_dim=64, partial_top_layers=4,
    c_input_scale="match",
    teacher=None, kd_docs=2000,
    gen_batch=16, seed=SEED,
)

EXPERIMENTS = {
    "S1_D_lorma":    dict(backbone="D", technique="lorma", lora_r=32, lora_alpha=64),
    "S2_B_houlsby":  dict(backbone="B", technique="adapter", adapter_dim=128),
    "S3_A4_kd":      dict(backbone="A", n_layers=4, technique="full", lr_backbone=1e-4,
                          teacher="S2_B_houlsby"),
    "S4_C_lora":     dict(backbone="C", technique="lora", peft_targets="all", lora_r=16, lora_alpha=32),
    "X1_B_pfeiffer":     dict(backbone="B", technique="pfeiffer", adapter_dim=128),
    "X2_B_houlsby_head": dict(backbone="B", technique="adapter", adapter_dim=128, doc_selection="head"),
}
RUN_EXPERIMENTS = ["S2_B_houlsby", "S1_D_lorma", "S3_A4_kd", "S4_C_lora"]
if os.environ.get("CS781_P6_RUN"):
    RUN_EXPERIMENTS = [x for x in os.environ["CS781_P6_RUN"].split(",") if x]

DECODING_CONFIGS = {
    "greedy_rep1.1_ng8": dict(repetition_penalty=1.1, no_repeat_ngram=8),
    "greedy_rep1.2_ng6": dict(repetition_penalty=1.2, no_repeat_ngram=6),
}
PREDICTION_KEY = "summary"

SMOKE_TEST = os.environ.get("CS781_P6_SMOKE", "0") == "1"
if SMOKE_TEST:
    DEFAULTS.update(tokens_per_batch=600, epochs=2, lora_r=4, lora_alpha=8, adapter_dim=8, kd_docs=20, gen_batch=4)
    for _cfg in EXPERIMENTS.values():
        if _cfg.get("n_layers"):
            _cfg["n_layers"] = 1
        for _k in ("lora_r", "lora_alpha", "adapter_dim"):
            _cfg.pop(_k, None)
    GEN_EVAL_DOCS = 8
    print("*** SMOKE TEST MODE: tiny settings, results are meaningless ***")

def experiment_config(name):
    cfg = dict(DEFAULTS)
    cfg.update(EXPERIMENTS[name])
    cfg["name"] = name
    return cfg

def submission_name(cfg, val_rouge_l):
    """e.g. phase6_S2_B-GQA-RoPE_HoulsbyAdapter_headtail_valRL0.2512.zip"""
    technique = {"lorma": "LoRMA", "lora": "LoRA", "adapter": "HoulsbyAdapter", "pfeiffer": "PfeifferAdapter",
                 "bitfit": "BitFit", "full": "FullFT", "partial": "PartialFT", "frozen": "Frozen"}[cfg["technique"]]
    if cfg["technique"] in ("lorma", "lora"):
        technique += f"-{cfg['peft_targets']}-r{cfg['lora_r']}"
    if cfg["teacher"]:
        technique = f"SeqKD-from-{cfg['teacher'].split('_')[0]}"
    model = MODEL_TAGS[cfg["backbone"]] + (f"-first{cfg['n_layers']}layers" if cfg["n_layers"] else "")
    return f"phase6_{cfg['name'].split('_')[0]}_{model}_{technique}_{cfg['doc_selection'].replace('_', '')}_valRL{val_rouge_l:.4f}.zip"

def restore_previous_outputs():
    """Copy a previous run of this notebook (attached with "Add Input") into the working folder, so finished
    experiments are re-used instead of trained again."""
    if not os.path.isdir(INPUT_ROOT):
        return
    copied = 0
    for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
        if os.path.basename(dirpath) == "cs781_phase6":
            for sub, _, files in os.walk(dirpath):
                for name in files:
                    src = os.path.join(sub, name)
                    dst = os.path.join(WORK_ROOT, os.path.relpath(src, dirpath))
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy2(src, dst)
                        copied += 1
            dirnames[:] = []
        elif dirpath[len(INPUT_ROOT):].count(os.sep) >= 6:
            dirnames[:] = []
    if copied:
        print(f"Restored {copied} file(s) from a previous Phase-6 run.")

restore_previous_outputs()
print("Working folder:", WORK_ROOT)
print("Experiments to run:", RUN_EXPERIMENTS)
for _name in RUN_EXPERIMENTS:
    print(f"  {_name:20s} -> {submission_name(experiment_config(_name), 0.0).replace('_valRL0.0000.zip', '_valRL<score>.zip')}")

Working folder: /kaggle/working/cs781_phase6
Experiments to run: ['S2_B_houlsby', 'S1_D_lorma', 'S3_A4_kd', 'S4_C_lora']
  S2_B_houlsby         -> phase6_S2_B-GQA-RoPE_HoulsbyAdapter_headtail_valRL<score>.zip
  S1_D_lorma           -> phase6_S1_D-GPT400M_LoRMA-qv-r32_headtail_valRL<score>.zip
  S3_A4_kd             -> phase6_S3_A-GPT220M-first4layers_SeqKD-from-S2_headtail_valRL<score>.zip
  S4_C_lora            -> phase6_S4_C-EncDec_LoRA-all-r16_headtail_valRL<score>.zip


## 3. Load the summarisation data and hold out a validation part

In [3]:
def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(obj, path):
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(path + ".tmp", path)

_HF_FILES = None

def phase6_file(name):
    """Local path of a Phase-6 file such as "train.json" (local folder, attached dataset, or Hugging Face download)."""
    global _HF_FILES
    candidates = []
    if LOCAL_DATA_DIR:
        candidates += sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "**", name), recursive=True))
    if os.path.isdir(INPUT_ROOT):
        for depth in range(0, 4):
            candidates += sorted(glob.glob(os.path.join(INPUT_ROOT, "*", *(["*"] * depth), PHASE6_DIR, "**", name), recursive=True))
    for c in candidates:
        if os.path.isfile(c):
            return c
    from huggingface_hub import hf_hub_download, list_repo_files
    if _HF_FILES is None:
        _HF_FILES = list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    matches = sorted((f for f in _HF_FILES if f.startswith(PHASE6_DIR + "/") and f.split("/")[-1] == name), key=len)
    if not matches:
        raise FileNotFoundError(f"{name} not found in {HF_DATASET_REPO}/{PHASE6_DIR}")
    print(f"downloading {matches[0]} ...", flush=True)
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=matches[0], repo_type="dataset", token=HF_TOKEN)

def read_cases(name, with_summary):
    """-> list of {id (exactly as in the file), document, summary or None}. Whitespace is normalised."""
    data = load_json_file(phase6_file(name))
    if isinstance(data, dict):
        data = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "document": v} for k, v in data.items()]
    out = []
    for i, d in enumerate(data):
        doc = d.get("document", d.get("text", d.get("judgment", "")))
        case = {"id": d.get("id", i), "document": " ".join(str(doc or "").split()), "summary": None}
        if with_summary:
            summ = d.get("summary", d.get("abstract"))
            if summ is None:
                raise ValueError(f"{name}: document {case['id']} has no summary")
            case["summary"] = " ".join(str(summ).split())
        out.append(case)
    return out

_all_train = read_cases("train.json", True)
CASES = {"test": read_cases("test.json", False)}
_ids = [json.dumps(c["id"]) for c in CASES["test"]]
assert len(set(_ids)) == len(_ids), "duplicate test ids"

_n_val = int(min(VAL_MAX, max(VAL_MIN, round(len(_all_train) * VAL_FRACTION))))
_perm = np.random.RandomState(SEED).permutation(len(_all_train))
CASES["val"] = [_all_train[i] for i in sorted(_perm[:_n_val])]
CASES["train"] = [_all_train[i] for i in sorted(_perm[_n_val:])]
del _all_train

print(f"{'split':6s} {'documents':>9s} {'doc words: median':>18s} {'p90':>8s} {'max':>9s} {'summary words: median':>22s} {'p90':>6s} {'max':>7s}")
for _s in ("train", "val", "test"):
    _d = [len(c["document"].split()) for c in CASES[_s]]
    _m = [len(c["summary"].split()) for c in CASES[_s]] if _s != "test" else None
    print(f"{_s:6s} {len(_d):9,} {int(np.median(_d)):18,} {int(np.percentile(_d, 90)):8,} {max(_d):9,} "
          + (f"{int(np.median(_m)):22,} {int(np.percentile(_m, 90)):6,} {max(_m):7,}" if _m else f"{'-':>22s}"))
_ratio = [len(c["summary"].split()) / max(1, len(c["document"].split())) for c in CASES["train"]]
print(f"summary length / document length (train): median {np.median(_ratio):.3f}")
print("Example summary (train):", CASES["train"][0]["summary"][:400])

downloading Phase_6_Summ/Train/train.json ...


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

downloading Phase_6_Summ/Test/test.json ...


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

split  documents  doc words: median      p90       max  summary words: median    p90     max
train      6,730              3,118    7,879   139,645                    637  1,498  27,768
val          300              3,450    7,431    90,410                    666  1,520   9,951
test         100              3,171    8,562    39,695                      -
summary length / document length (train): median 0.218
Example summary (train): The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be paid for the cement supplied by the marketing company, which had the discretion to fix the sale price. For the 1950 51, 1951 52 and 1952 


## 4. Phase-1 tokenizer (encode and decode) and document selection

In [4]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
PAD_ID = 0

class BPETokenizer:
    """Loads the byte-level BPE tokenizer saved by a Phase-1 notebook (tokenizer.json) and encodes text with the
    same merge rules. Only loading and encoding are needed here; the tokenizer is NOT retrained."""

    def __init__(self, directory):
        with open(os.path.join(directory, "tokenizer.json"), "r", encoding="utf-8") as f:
            data = json.load(f)
        self.token_to_id = {str(k): int(v) for k, v in data["token_to_id"].items()}
        self.merge_ranks = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        self.unk_id = self.token_to_id["<unk>"]
        self.eow_id = self.token_to_id["</w>"]
        self.vocab_size = len(self.token_to_id)
        self._cache = {}
        self.id_to_token = {i: t for t, i in self.token_to_id.items()}
        self._merge_map = {a + b: (a, b) for a, b in (tuple(m) for m in data["merges"])}
        self._byte_tokens = {str(i) for i in range(256)}
        self.bos_id = self.token_to_id["<s>"]
        self.eos_id = self.token_to_id["</s>"]
        self.pad_id = self.token_to_id["<pad>"]

    def _merge(self, text):
        tokens = [str(b) for b in text.encode("utf-8")]
        while len(tokens) > 1:
            best_rank, best_index = None, -1
            for i in range(len(tokens) - 1):
                rank = self.merge_ranks.get((tokens[i], tokens[i + 1]))
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_rank, best_index = rank, i
            if best_index < 0:
                break
            tokens[best_index:best_index + 2] = [tokens[best_index] + tokens[best_index + 1]]
        return [self.token_to_id.get(t, self.unk_id) for t in tokens]

    def encode_piece(self, piece):
        """Token ids of one piece of text WITHOUT the end-of-word token."""
        ids = self._cache.get(piece)
        if ids is None:
            ids = []
            for k in range(0, len(piece), 200):
                ids.extend(self._merge(piece[k:k + 200]))
            if len(self._cache) < 2_000_000:
                self._cache[piece] = ids
        return ids

    def encode(self, text, max_tokens=None):
        ids = []
        for word in str(text).split():
            ids.extend(self.encode_piece(word))
            ids.append(self.eow_id)
            if max_tokens is not None and len(ids) >= max_tokens:
                break
        return ids if max_tokens is None else ids[:max_tokens]

    def _bytes(self, token):
        if token in self._merge_map:
            a, b = self._merge_map[token]
            return self._bytes(a) + self._bytes(b)
        if token in self._byte_tokens:
            return bytes([int(token)])
        return b""

    def decode(self, ids):
        words, current = [], bytearray()
        for i in ids:
            token = self.id_to_token.get(int(i))
            if token is None or token in SPECIAL_TOKENS:
                continue
            if token == "</w>":
                if current:
                    words.append(current.decode("utf-8", errors="replace"))
                    current = bytearray()
                continue
            current.extend(self._bytes(token))
        if current:
            words.append(current.decode("utf-8", errors="replace"))
        return " ".join(words)

def select_document_tokens(ids, budget, mode):
    """Long documents: mode "head" keeps the first `budget` tokens; "head_tail" keeps the first 60% and the last 40%
    of the budget (judgments state the facts at the start and the decision at the end)."""
    if len(ids) <= budget:
        return list(ids)
    if mode == "head":
        return list(ids[:budget])
    head = int(round(budget * 0.6))
    return list(ids[:head]) + list(ids[len(ids) - (budget - head):])

## 5. Metrics: ROUGE-1, ROUGE-L, redundancy (and BERTScore if installed)

In [5]:
def _tokens(text):
    return re.findall(r"\w+", str(text).lower())

def rouge_1_f1(cand, ref):
    c, r = Counter(_tokens(cand)), Counter(_tokens(ref))
    overlap = sum((c & r).values())
    if not overlap:
        return 0.0
    p, rec = overlap / sum(c.values()), overlap / sum(r.values())
    return 2 * p * rec / (p + rec)

def rouge_l_f1(cand, ref, cap=1500):
    c, r = _tokens(cand)[:cap], _tokens(ref)[:cap]
    if not c or not r:
        return 0.0
    index = defaultdict(list)
    for j, w in enumerate(r):
        index[w].append(j)
    prev = np.zeros(len(r) + 1, dtype=np.int32)
    for w in c:
        cur = prev.copy()
        for j in index.get(w, ()):
            cur[j + 1] = prev[j] + 1
        np.maximum.accumulate(cur, out=cur)
        prev = cur
    lcs = int(prev[-1])
    if not lcs:
        return 0.0
    p, rec = lcs / len(c), lcs / len(r)
    return 2 * p * rec / (p + rec)

try:
    import importlib.util as _ilu
    HAVE_BERTSCORE = _ilu.find_spec("bert_score") is not None
except Exception:
    HAVE_BERTSCORE = False

def bertscore_f1(cands, refs):
    """Mean BERTScore F1 (only if the package is installed; returns None otherwise or on any error)."""
    if not HAVE_BERTSCORE:
        return None
    try:
        from bert_score import score as _bs
        _, _, f = _bs(list(cands), list(refs), lang="en", verbose=False, batch_size=16,
                      device="cuda" if DEVICE.type == "cuda" else "cpu")
        return float(f.mean())
    except Exception as e:
        print("BERTScore not available here (" + type(e).__name__ + "); using ROUGE only.")
        return None

def summary_scores(cands, refs):
    r1 = float(np.mean([rouge_1_f1(c, r) for c, r in zip(cands, refs)]))
    rl = float(np.mean([rouge_l_f1(c, r) for c, r in zip(cands, refs)]))
    words = [len(c.split()) for c in cands]
    ref_words = [len(r.split()) for r in refs]
    rep = []
    for c in cands:
        w = _tokens(c)
        grams = [tuple(w[i:i + 4]) for i in range(len(w) - 3)]
        rep.append(1 - len(set(grams)) / len(grams) if grams else 0.0)
    return {"rouge1_f1": r1, "rougeL_f1": rl, "words_mean": float(np.mean(words)), "reference_words_mean": float(np.mean(ref_words)),
            "repeated_4gram_share": float(np.mean(rep))}

## 6. Phase-1 models (C now with its decoder)

In [6]:
ARCH_OF_TYPE = {"decoder-only": "A", "decoder-only-gqa": "B", "encoder-decoder": "C", "efficient-local-mqa": "D-alibi"}
ARCH_DESCRIPTION = {"A": "GPT decoder", "B": "GQA/RoPE decoder", "C": "encoder-decoder", "D-alibi": "local MQA/ALiBi decoder"}

def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def discover_phase1_checkpoints(root):
    """Finds the models written by the Phase-1 notebooks:
      <...>/<ARCH>/legal-model-final/{model.pt, config.json, tokenizer.json}       (normal case), or
      <...>/<ARCH>/best_model.pt + <...>/common/bpe_tokenizer_<ARCH>/tokenizer.json  (run stopped before the final save)."""
    found = []
    if not os.path.isdir(root):
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) >= 12:
            dirnames[:] = []
        try:
            if os.path.basename(dirpath) == "legal-model-final" and "model.pt" in filenames and "tokenizer.json" in filenames:
                config = load_json_file(os.path.join(dirpath, "config.json"))
                history_path = os.path.join(dirpath, "training_history.json")
                model_file, tokenizer_dir = os.path.join(dirpath, "model.pt"), dirpath
                if os.path.exists(history_path):
                    eval_loss = float(load_json_file(history_path).get("best_eval_loss", "inf"))
                else:
                    eval_loss = float(torch_load(model_file).get("best_eval_loss", "inf"))
            elif "best_model.pt" in filenames and "legal-model-final" not in dirnames:
                common = os.path.join(os.path.dirname(dirpath), "common")
                tokenizer_dirs = [d for d in sorted(glob.glob(os.path.join(common, "bpe_tokenizer_" + os.path.basename(dirpath) + "*")))
                                  if os.path.exists(os.path.join(d, "tokenizer.json"))]
                if not tokenizer_dirs:
                    continue
                saved = torch_load(os.path.join(dirpath, "best_model.pt"))
                config, eval_loss = saved["config"], float(saved.get("eval_loss", "inf"))
                del saved
                model_file, tokenizer_dir = os.path.join(dirpath, "best_model.pt"), tokenizer_dirs[-1]
            else:
                continue
            arch = ARCH_OF_TYPE.get(config.get("type")) or str(config.get("architecture", "?"))[:1]
        except Exception as e:
            print("Skipping", dirpath, "(" + type(e).__name__ + ")")
            continue
        found.append({"arch": arch, "dir": dirpath, "model_file": model_file, "tokenizer_dir": tokenizer_dir, "config": config,
                      "eval_loss": eval_loss, "size_mb": os.path.getsize(model_file) / 1e6})
    return found

PHASE1_FOUND = discover_phase1_checkpoints(INPUT_ROOT)
PHASE1 = {}
print(f"Phase-1 checkpoints found under {INPUT_ROOT}: {len(PHASE1_FOUND)}")
for c in sorted(PHASE1_FOUND, key=lambda c: (c["arch"], c["eval_loss"])):
    print(f"  {ARCH_DESCRIPTION.get(c['arch'], c['arch']):24s} val loss {c['eval_loss']:.4f} | {c['size_mb']:.0f} MB | {c['dir']}")
_chosen = {}
for slot, spec in sorted(PHASE1_MODELS.items(), key=lambda kv: kv[1]["pick"] == "other"):
    arch, choice = spec["architecture"], spec["pick"]
    candidates = [c for c in PHASE1_FOUND if c["arch"] == arch]
    if choice == "other":
        taken = [_chosen[a]["eval_loss"] for a in _chosen if _chosen[a]["arch"] == arch]
        candidates = [c for c in candidates if all(abs(c["eval_loss"] - t) > 1e-6 for t in taken)]
        pick = max(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    else:
        if choice not in ("best", "auto"):
            candidates = [c for c in candidates if choice.lower() in c["dir"].lower()]
        pick = min(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    if pick is not None:
        PHASE1[slot] = _chosen[slot] = dict(pick, slot=slot)
print("Your Phase-1 models:")
for slot in PHASE1_MODELS:
    if slot in PHASE1:
        e = PHASE1[slot]
        print(f"  {slot}: {ARCH_DESCRIPTION.get(e['arch'], e['arch'])}, validation loss {e['eval_loss']:.4f} | {e['dir']}")
    else:
        print(f"  {slot}: NOT FOUND - attach the output of that Phase-1 notebook with 'Add Input'")

TOKENIZERS = {}

def tokenizer_for(slot):
    path = PHASE1[slot]["tokenizer_dir"]
    if path not in TOKENIZERS:
        TOKENIZERS[path] = BPETokenizer(path)
    return TOKENIZERS[path]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x * rms.to(x.dtype)) * self.weight

class AttentionA(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.qkv = nn.Linear(d, 3 * d, bias=True)
        self.out = nn.Linear(d, d, bias=True)

    def forward(self, x):
        b, t, c = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [z.view(b, t, self.n_heads, self.head_dim).transpose(1, 2) for z in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockA(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = AttentionA(d, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.ln2(x)))))

class BackboneA(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(cfg["context_length"], d)
        self.blocks = nn.ModuleList([BlockA(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None]
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.ln_f(x))
        return states

def rotate_half(x):
    return torch.stack((-x[..., 1::2], x[..., ::2]), dim=-1).flatten(-2)

class AttentionB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, max_len, dropout):
        super().__init__()
        self.q_heads, self.kv_heads, self.head_dim, self.dropout = n_heads, n_kv_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
        self.register_buffer("cos", torch.cos(freqs).repeat_interleave(2, dim=-1), persistent=False)
        self.register_buffer("sin", torch.sin(freqs).repeat_interleave(2, dim=-1), persistent=False)

    def forward(self, x):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.cos[:t].to(q.dtype)[None, None], self.sin[:t].to(q.dtype)[None, None]
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        repeat = self.q_heads // self.kv_heads
        if repeat > 1:
            k, v = k.repeat_interleave(repeat, dim=1), v.repeat_interleave(repeat, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.up = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class BlockB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, d_ff, max_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionB(d, n_heads, n_kv_heads, max_len, dropout)
        self.norm2 = RMSNorm(d)
        self.mlp = SwiGLU(d, d_ff)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        return x + self.dropout(self.mlp(self.norm2(x)))

class BackboneB(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockB(d, cfg["n_head"], cfg["n_kv_head"], cfg["d_ff"], cfg["context_length"], dropout)
                                     for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.norm_f(x))
        return states

class AttentionC(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.o_proj = nn.Linear(d, d)

    def forward(self, x, attn_mask=None):
        b, t, c = x.shape
        q, k, v = [proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
                   for proj in (self.q_proj, self.k_proj, self.v_proj)]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class EncoderBlockC(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = AttentionC(d, n_heads, dropout)
        self.ln1 = nn.LayerNorm(d)
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)
        self.ln2 = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        x = self.ln1(x + self.dropout(self.self_attn(x, attn_mask)))
        return self.ln2(x + self.dropout(self.ff2(F.gelu(self.ff1(x)))))

class BackboneC(nn.Module):
    bidirectional = True

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["encoder_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.encoder = nn.ModuleList([EncoderBlockC(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        pe = torch.zeros(self.max_tokens, d)
        position = torch.arange(0, self.max_tokens).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def layers(self):
        return self.encoder

    embedding_state = False

    input_scale = 1.0

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids) * self.input_scale
        states = []
        if self.embedding_state:
            scale = self.token_embedding.weight.detach().float().pow(2).mean().sqrt().clamp_min(1e-12)
            states.append(F.layer_norm(x / scale.to(x.dtype), (self.d_model,)))
        x = x + self.pe[:x.size(1)].to(x.dtype)[None]
        attn_mask = None if pad_mask is None else pad_mask[:, None, None, :]
        for block in self.encoder:
            x = block(x, attn_mask)
            states.append(x)
        return states

class AttentionD(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, attn_bias):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        v = self.v_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias.to(q.dtype),
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockD(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionD(d, n_heads, dropout)
        self.norm2 = RMSNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_bias):
        x = x + self.dropout(self.attn(self.norm1(x), attn_bias))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.norm2(x)))))

class BackboneD(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens, self.window = d, cfg["context_length"], cfg["local_window"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockD(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)
        n = cfg["n_head"]
        self.register_buffer("slopes", 2.0 ** (-(8.0 * torch.arange(n) / n)), persistent=False)

    def layers(self):
        return self.blocks

    def attention_bias(self, t, device):
        pos = torch.arange(t, device=device, dtype=torch.float32)
        distance = pos[:, None] - pos[None, :]
        blocked = (distance < 0) | (distance > self.window - 1)
        bias = -self.slopes.to(device)[:, None, None] * distance[None]
        return bias.masked_fill(blocked[None], float("-inf"))[None]

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        attn_bias = self.attention_bias(input_ids.size(1), input_ids.device)
        states = []
        for block in self.blocks:
            x = block(x, attn_bias)
            states.append(self.norm_f(x))
        return states

BACKBONE_CLASSES = {"A": BackboneA, "B": BackboneB, "C": BackboneC, "D-alibi": BackboneD}

@torch.no_grad()
def encoder_token_check(backbone, length=256):
    """Does the Phase-1 encoder output depend on the input tokens? Two different random token sequences are encoded;
    if the last-layer states at the same positions are almost identical, the encoder only encodes positions."""
    was_training = backbone.training
    backbone.eval()
    g = torch.Generator().manual_seed(0)
    vocab = backbone.token_embedding.weight.shape[0]
    length = min(length, backbone.max_tokens)
    ids = torch.randint(4, vocab, (2, length), generator=g)
    flag, backbone.embedding_state = backbone.embedding_state, False
    last = backbone(ids)[-1].float()
    backbone.embedding_state = flag
    same_position = F.cosine_similarity(last[0], last[1], dim=-1).mean().item()
    other_position = F.cosine_similarity(last[0], last[0].roll(1, 0), dim=-1).mean().item()
    emb_rms = backbone.token_embedding.weight.float().pow(2).mean().sqrt().item()
    pe_rms = backbone.pe[:length].float().pow(2).mean().sqrt().item()
    print(f"Encoder check: token-embedding RMS {emb_rms:.4f} vs position-encoding RMS {pe_rms:.4f} | last layer, "
          f"cosine similarity of two DIFFERENT inputs at the same position {same_position:.3f} "
          f"(neighbouring positions of one input {other_position:.3f}; close to 1.0 = the output ignores the tokens)")
    backbone.train(was_training)

def load_backbone(slot, n_layers=None, dropout=0.1, verbose=True):
    """Build the architecture and load its Phase-1 weights (optionally only the first n layers)."""
    entry = PHASE1[slot]
    arch = entry["arch"]
    cfg = entry["config"]
    state = torch_load(entry["model_file"])["model"]
    total_layers = cfg["encoder_layers"] if arch == "C" else cfg["n_layer"]
    n_layers = total_layers if n_layers is None else min(int(n_layers), total_layers)
    vocab_size = state["token_embedding.weight"].shape[0]
    backbone = BACKBONE_CLASSES[arch](cfg, vocab_size, n_layers, dropout)
    wanted = set(backbone.state_dict().keys())
    missing = wanted - set(state.keys())
    if missing:
        raise RuntimeError(f"Phase-1 checkpoint for {slot} lacks weights: {sorted(missing)[:5]} ...")
    backbone.load_state_dict({k: v for k, v in state.items() if k in wanted}, strict=True)
    if verbose and arch == "C":
        encoder_token_check(backbone)
    if verbose:
        unused = len(state) - len(wanted)
        print(f"Loaded Phase-1 model {slot} ({ARCH_DESCRIPTION.get(arch, arch)}): {n_layers}/{total_layers} layers, "
              f"{sum(p.numel() for p in backbone.parameters()) / 1e6:.2f}M parameters"
              + (f" ({unused} weight tensors not needed, e.g. removed layers or the decoder)" if unused else ""))
    return backbone

class DecoderBlockC(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = AttentionC(d, n_heads, dropout)
        self.cross_attn = AttentionC(d, n_heads, dropout)
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.ln3 = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)

class Seq2SeqC(nn.Module):
    def __init__(self, cfg, vocab_size, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.n_heads = d, cfg["n_head"]
        self.encoder_model = BackboneC(cfg, vocab_size, cfg["encoder_layers"], dropout)
        self.token_embedding = self.encoder_model.token_embedding
        self.decoder = nn.ModuleList([DecoderBlockC(d, cfg["n_head"], cfg["d_ff"], dropout)
                                      for _ in range(cfg.get("decoder_layers", cfg["encoder_layers"]))])
        self.max_src, self.max_tgt = cfg["encoder_length"], cfg.get("decoder_length", cfg["encoder_length"])
        self.register_buffer("tgt_pe", self.encoder_model.pe.new_zeros(0), persistent=False)
        pe = torch.zeros(self.max_tgt, d)
        position = torch.arange(0, self.max_tgt).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.tgt_pe = pe

    def layers(self):
        return list(self.encoder_model.encoder) + list(self.decoder)

def load_seq2seq_c(slot, dropout=0.1, verbose=True):
    entry = PHASE1[slot]
    if entry["arch"] != "C":
        raise ValueError("load_seq2seq_c is for the encoder-decoder model C")
    cfg = entry["config"]
    state = torch_load(entry["model_file"])["model"]
    vocab_size = state["token_embedding.weight"].shape[0]
    model = Seq2SeqC(cfg, vocab_size, dropout)
    renamed = {}
    for k, v in state.items():
        if k.startswith("encoder."):
            renamed["encoder_model." + k] = v
        elif k == "token_embedding.weight":
            renamed["encoder_model.token_embedding.weight"] = v
            renamed["token_embedding.weight"] = v
        elif k.startswith("decoder."):
            renamed[k] = v
    wanted = set(model.state_dict().keys())
    missing = wanted - set(renamed)
    if missing:
        raise RuntimeError(f"Phase-1 checkpoint for {slot} lacks weights: {sorted(missing)[:5]} ...")
    model.load_state_dict({k: v for k, v in renamed.items() if k in wanted}, strict=True)
    if verbose:
        encoder_token_check(model.encoder_model)
        print(f"Loaded Phase-1 model {slot} (encoder-decoder): {len(model.encoder_model.encoder)} encoder + "
              f"{len(model.decoder)} decoder layers, {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
    return model

Phase-1 checkpoints found under /kaggle/input: 4
  GPT decoder              val loss 1.3086 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-D/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GPT decoder              val loss 1.7840 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GQA/RoPE decoder         val loss 1.6521 | 264 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-B/cs781_phase1/B_GQA_RoPE_RMSNorm_SwiGLU/legal-model-final
  encoder-decoder          val loss 3.0858 | 422 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-C/cs781_phase1/C_EncoderDecoder/legal-model-final
Your Phase-1 models:
  A: GPT decoder, validation loss 1.7840 | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  B: GQA/RoPE decoder, validation loss 1.6521

## 7. Fine-tuning and adaptation techniques

| Technique | What is trained |
|---|---|
| `adapter` | Houlsby bottleneck adapters after attention and feed-forward (+ norms) |
| `pfeiffer` | one bottleneck adapter per block, after the feed-forward only (+ norms) |
| `lorma` | low-rank multiplicative update (I + (α/r)·BA)·W0 (LoRMA+), computed in float32 |
| `lora` | low-rank additive update W0 + (α/r)·BA |
| `full` | every weight (the distilled student) |

In [7]:
class LoRALinear(nn.Module):
    """LoRA (Hu et al.): additive low-rank update,  W = W0 + (alpha / r) * B A.   B starts at zero.
    With scaling="rs" it is rsLoRA:  W = W0 + (alpha / sqrt(r)) * B A."""

    def __init__(self, base, r, alpha, rows=None, scaling="standard"):
        super().__init__()
        self.base, self.scale = base, (alpha / math.sqrt(r) if scaling == "rs" else alpha / r)
        self.rows = rows or [(0, base.out_features)]
        self.lora_A = nn.ParameterList([nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
                                        for _ in self.rows])
        self.lora_B = nn.ParameterList([nn.Parameter(torch.zeros(e - s, r)) for s, e in self.rows])

    def forward(self, x):
        y = self.base(x)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lora_A, self.lora_B):
            if s > cursor:
                parts.append(y[..., cursor:s])
            parts.append(y[..., s:e] + self.scale * F.linear(F.linear(x, A.to(x.dtype)), B.to(x.dtype)))
            cursor = e
        if cursor < y.size(-1):
            parts.append(y[..., cursor:])
        return parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)

_PERM_INDEX = {}

class LoRMALinear(nn.Module):
    """LoRMA (Low-Rank Multiplicative Adaptation): the pretrained weight is MULTIPLIED by a learned matrix.

      "plus" (LoRMA+):   W = (I + (alpha / r) * B A) W0     B starts at zero, so the multiplier starts as the identity.
      "pi"   (LoRMA-pi): W = P(B A) W0, where P shifts row i of (B A) cyclically by i positions, which makes a
                         low-rank product full rank. B's first column is ones and A[0, 0] = 1 (rest of A zero), so
                         P(B A) starts as the identity.
    Because the multiplier is on the left of W0, it acts on the layer's OUTPUT: h = W0 x, y = M h + bias.
    """

    def __init__(self, base, r, alpha, rows=None, variant="plus"):
        super().__init__()
        self.base, self.scale, self.variant = base, alpha / r, variant
        self.rows = rows or [(0, base.out_features)]
        self.lorma_A, self.lorma_B = nn.ParameterList(), nn.ParameterList()
        for s, e in self.rows:
            m = e - s
            if variant == "plus":
                A, B = torch.randn(r, m) / math.sqrt(m), torch.zeros(m, r)
            elif variant == "pi":
                A, B = torch.zeros(r, m), torch.randn(m, r) * 0.02
                A[0, 0] = 1.0
                B[:, 0] = 1.0
            else:
                raise ValueError("lorma_variant must be 'plus' or 'pi'")
            self.lorma_A.append(nn.Parameter(A))
            self.lorma_B.append(nn.Parameter(B))

    @staticmethod
    def _permuted(BA):
        m = BA.size(0)
        key = (m, BA.device)
        if key not in _PERM_INDEX:
            i = torch.arange(m, device=BA.device)
            _PERM_INDEX[key] = (i[None, :] - i[:, None]) % m
        return torch.gather(BA, 1, _PERM_INDEX[key])

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lorma_A, self.lorma_B):
            if s > cursor:
                parts.append(h[..., cursor:s])
            hs = h[..., s:e]
            if self.variant == "plus":
                upd = self.scale * F.linear(F.linear(hs.float(), A.float()), B.float())
                parts.append((hs.float() + upd).to(hs.dtype))
            else:
                parts.append(F.linear(hs, self._permuted(B @ A).to(hs.dtype)))
            cursor = e
        if cursor < h.size(-1):
            parts.append(h[..., cursor:])
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        return y if self.base.bias is None else y + self.base.bias

class AdapterLinear(nn.Module):
    """Houlsby-style bottleneck adapter placed on a sub-layer's output, before the residual connection:
    y = h + up(GELU(down(h))),  h = base(x).   `up` starts at zero, so the adapter starts as the identity."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        self.adapter_down = nn.Linear(base.out_features, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, base.out_features)
        nn.init.normal_(self.adapter_down.weight, std=0.02)
        nn.init.zeros_(self.adapter_down.bias)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.adapter_up(F.gelu(self.adapter_down(h)))

class IA3Linear(nn.Module):
    """IA3 (Liu et al., "(IA)^3"): learned vectors that RESCALE activations, initialised to ones (= no change).
    mode "out": rescale the layer's output (used for attention keys and values; `rows` limits it to part of a fused
    matrix).  mode "in": rescale the layer's input (used for the feed-forward hidden activations, before fc2)."""

    def __init__(self, base, mode, rows=None):
        super().__init__()
        self.base, self.mode = base, mode
        if mode == "in":
            self.ia3_scale = nn.Parameter(torch.ones(base.in_features))
            self.rows = None
        else:
            self.rows = rows or [(0, base.out_features)]
            index = torch.cat([torch.arange(s, e) for s, e in self.rows])
            self.register_buffer("row_index", index, persistent=False)
            self.ia3_scale = nn.Parameter(torch.ones(len(index)))

    def forward(self, x):
        if self.mode == "in":
            return self.base(x * self.ia3_scale.to(x.dtype))
        y = self.base(x)
        full = torch.ones(y.size(-1), device=y.device, dtype=y.dtype).scatter(0, self.row_index, self.ia3_scale.to(y.dtype))
        return y * full

QV_TARGETS = {"A": ["attn.qkv"], "B": ["attn.q_proj", "attn.v_proj"], "C": ["self_attn.q_proj", "self_attn.v_proj"],
              "D-alibi": ["attn.q_proj", "attn.v_proj"]}
ALL_TARGETS = {"A": ["attn.qkv", "attn.out", "fc1", "fc2"],
               "B": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "mlp.gate", "mlp.up", "mlp.down"],
               "C": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj", "ff1", "ff2",
                     "cross_attn.q_proj", "cross_attn.k_proj", "cross_attn.v_proj", "cross_attn.o_proj"],
               "D-alibi": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "fc1", "fc2"]}
SUBLAYER_OUTPUTS = {"A": ["attn.out", "fc2"], "B": ["attn.o_proj", "mlp.down"], "C": ["self_attn.o_proj", "cross_attn.o_proj", "ff2"],
                    "D-alibi": ["attn.o_proj", "fc2"]}
IA3_TARGETS = {"A": [("attn.qkv", "out"), ("fc2", "in")], "B": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("mlp.down", "in")],
               "C": [("self_attn.k_proj", "out"), ("self_attn.v_proj", "out"), ("ff2", "in")],
               "D-alibi": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("fc2", "in")]}
NEW_WEIGHT_MARKERS = ("lora_", "lorma_", "adapter_", "ia3_")

FFN_OUTPUTS = {"A": ["fc2"], "B": ["mlp.down"], "C": ["ff2"], "D-alibi": ["fc2"]}

def _has(module, path):
    for part in path.split("."):
        if not hasattr(module, part):
            return False
        module = getattr(module, part)
    return True

def _get(module, path):
    for part in path.split("."):
        module = getattr(module, part)
    return module

def _set(module, path, value):
    parts = path.split(".")
    for part in parts[:-1]:
        module = getattr(module, part)
    setattr(module, parts[-1], value)

def apply_technique(backbone, arch, cfg):
    """Freeze the pretrained model, then make trainable exactly what the chosen technique allows."""
    technique = cfg["technique"]
    for p in backbone.parameters():
        p.requires_grad = False
    blocks = backbone.layers()
    norm_types = (nn.LayerNorm, RMSNorm)
    if technique == "full":
        for p in backbone.parameters():
            p.requires_grad = True
    elif technique == "partial":
        for block in list(blocks)[-int(cfg["partial_top_layers"]):]:
            for p in block.parameters():
                p.requires_grad = True
        for name in ("ln_f", "norm_f"):
            if hasattr(backbone, name):
                for p in getattr(backbone, name).parameters():
                    p.requires_grad = True
    elif technique == "bitfit":
        for name, p in backbone.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique in ("lora", "lorma"):
        targets = (QV_TARGETS if cfg["peft_targets"] == "qv" else ALL_TARGETS)[arch]
        for block in blocks:
            for path in targets:
                if not _has(block, path):
                    continue
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv" and cfg["peft_targets"] == "qv":
                    d = base.in_features
                    rows = [(0, d), (2 * d, 3 * d)]
                if technique == "lora":
                    wrapped = LoRALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg.get("lora_scaling", "standard"))
                else:
                    wrapped = LoRMALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg["lorma_variant"])
                _set(block, path, wrapped)
    elif technique == "ia3":
        for block in blocks:
            for path, mode in IA3_TARGETS[arch]:
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv":
                    d = base.in_features
                    rows = [(d, 2 * d), (2 * d, 3 * d)]
                _set(block, path, IA3Linear(base, mode, rows))
    elif technique in ("adapter", "pfeiffer"):
        for block in blocks:
            for path in (SUBLAYER_OUTPUTS if technique == "adapter" else FFN_OUTPUTS)[arch]:
                if _has(block, path):
                    _set(block, path, AdapterLinear(_get(block, path), cfg["adapter_dim"]))
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique != "frozen":
        raise ValueError("Unknown technique: " + str(technique))
    for name, p in backbone.named_parameters():
        if any(marker in name for marker in NEW_WEIGHT_MARKERS):
            p.requires_grad = True
    return backbone

## 8. Summarisation models and generation (key/value cache, batched)

In [8]:
class SummModel(nn.Module):
    def __init__(self, cfg, verbose=True):
        super().__init__()
        self.slot = cfg["backbone"]
        self.arch = PHASE1[self.slot]["arch"]
        if self.arch == "D-alibi":
            raise ValueError("the local-attention ALiBi model is not used in this phase")
        if self.arch == "C":
            self.net = load_seq2seq_c(self.slot, cfg["dropout"], verbose=verbose)
            scale = cfg.get("c_input_scale", 1.0)
            if scale == "match":
                emb_rms = self.net.token_embedding.weight.detach().float().pow(2).mean().sqrt().item()
                pe_rms = self.net.encoder_model.pe.float().pow(2).mean().sqrt().item()
                scale = max(1.0, pe_rms / max(emb_rms, 1e-8))
            self.net.encoder_model.input_scale = float(scale)
            if verbose and float(scale) != 1.0:
                print(f"Model C: encoder input embeddings scaled by {float(scale):.1f}")
            self.context = None
        else:
            self.net = load_backbone(self.slot, cfg["n_layers"], cfg["dropout"], verbose=verbose)
            self.context = self.net.max_tokens
        apply_technique(self.net, self.arch, cfg)

    @property
    def embedding(self):
        return self.net.token_embedding.weight

def _chunked_ce(hidden, emb, targets, chunk=1024):
    """Cross-entropy of the selected positions, the vocabulary projection done in chunks with recomputation (the
    full [tokens x vocabulary] logit matrix of a 2,048-token batch would not fit in memory). -> summed loss."""
    total = hidden.new_zeros((), dtype=torch.float32)
    for i in range(0, hidden.size(0), chunk):
        total = total + torch.utils.checkpoint.checkpoint(_ce_sum, hidden[i:i + chunk], emb, targets[i:i + chunk],
                                                          use_reentrant=False)
    return total

def _ce_sum(h, emb, t):
    return F.cross_entropy(F.linear(h.float(), emb.float()), t, reduction="sum")

def _attention(q, k, v, mask, dropout, training, causal=False):
    if mask is None:
        return F.scaled_dot_product_attention(q, k, v, dropout_p=dropout if training else 0.0, is_causal=causal)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dropout if training else 0.0)

def decoder_forward(net, arch, ids, positions, mask, cache):
    """ids/positions [B, T]; mask [B, 1, T, L] (True = may attend) or None (plain causal); cache: list of (k, v) per
    layer or None. -> final hidden states [B, T, d] (after the last norm), new cache."""
    new_cache = []
    if arch == "A":
        x = net.token_embedding(ids) + net.position_embedding(positions)
        for i, blk in enumerate(net.blocks):
            a = blk.attn
            b, t, c = x.shape
            q, k, v = a.qkv(blk.ln1(x)).chunk(3, dim=-1)
            q, k, v = [z.view(b, t, a.n_heads, a.head_dim).transpose(1, 2) for z in (q, k, v)]
            if cache is not None and cache[i] is not None:
                k, v = torch.cat([cache[i][0], k], 2), torch.cat([cache[i][1], v], 2)
            new_cache.append((k, v))
            y = _attention(q, k, v, mask, a.dropout, net.training, causal=mask is None)
            x = x + blk.dropout(a.out(y.transpose(1, 2).contiguous().view(b, t, c)))
            x = x + blk.dropout(blk.fc2(F.gelu(blk.fc1(blk.ln2(x)))))
        return net.ln_f(x), new_cache
    x = net.token_embedding(ids)
    for i, blk in enumerate(net.blocks):
        a = blk.attn
        b, t, c = x.shape
        h = blk.norm1(x)
        q = a.q_proj(h).view(b, t, a.q_heads, a.head_dim).transpose(1, 2)
        k = a.k_proj(h).view(b, t, a.kv_heads, a.head_dim).transpose(1, 2)
        v = a.v_proj(h).view(b, t, a.kv_heads, a.head_dim).transpose(1, 2)
        cos, sin = a.cos[positions].to(q.dtype)[:, None], a.sin[positions].to(q.dtype)[:, None]
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        if cache is not None and cache[i] is not None:
            k, v = torch.cat([cache[i][0], k], 2), torch.cat([cache[i][1], v], 2)
        new_cache.append((k, v))
        rep = a.q_heads // a.kv_heads
        kk, vv = (k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1)) if rep > 1 else (k, v)
        y = _attention(q, kk, vv, mask, a.dropout, net.training, causal=mask is None)
        x = x + blk.dropout(a.o_proj(y.transpose(1, 2).contiguous().view(b, t, c)))
        x = x + blk.dropout(blk.mlp(blk.norm2(x)))
    return net.norm_f(x), new_cache

def c_encode(net, src, src_mask):
    memory = net.encoder_model(src, src_mask)[-1]
    kv = []
    for blk in net.decoder:
        a = blk.cross_attn
        b, s, c = memory.shape
        k = a.k_proj(memory).view(b, s, a.n_heads, a.head_dim).transpose(1, 2)
        v = a.v_proj(memory).view(b, s, a.n_heads, a.head_dim).transpose(1, 2)
        kv.append((k, v))
    return kv

def c_decode(net, tgt, start, memory_kv, src_mask, cache):
    """tgt [B, T] decoder input ids starting at position `start`; -> hidden [B, T, d], new self-attention cache."""
    y = net.token_embedding(tgt) + net.tgt_pe[start:start + tgt.size(1)].to(net.token_embedding.weight.dtype)[None]
    cross_mask = src_mask[:, None, None, :]
    new_cache = []
    for i, blk in enumerate(net.decoder):
        b, t, c = y.shape
        a = blk.self_attn
        q, k, v = [p(y).view(b, t, a.n_heads, a.head_dim).transpose(1, 2) for p in (a.q_proj, a.k_proj, a.v_proj)]
        if cache is not None and cache[i] is not None:
            k, v = torch.cat([cache[i][0], k], 2), torch.cat([cache[i][1], v], 2)
        new_cache.append((k, v))
        causal = cache is None or cache[i] is None
        z = _attention(q, k, v, None, a.dropout, net.training, causal=causal and t > 1)
        y = blk.ln1(y + blk.dropout(a.o_proj(z.transpose(1, 2).contiguous().view(b, t, c))))
        a = blk.cross_attn
        q = a.q_proj(y).view(b, t, a.n_heads, a.head_dim).transpose(1, 2)
        z = _attention(q, memory_kv[i][0], memory_kv[i][1], cross_mask, a.dropout, net.training)
        y = blk.ln2(y + blk.dropout(a.o_proj(z.transpose(1, 2).contiguous().view(b, t, c))))
        y = blk.ln3(y + blk.dropout(blk.ff2(F.gelu(blk.ff1(y)))))
    return y, new_cache

def training_loss(model, batch):
    """Mean cross-entropy over the summary tokens of a batch. -> (loss, number of target tokens)."""
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        if model.arch == "C":
            kv = c_encode(model.net, batch["src"], batch["src_mask"])
            hidden, _ = c_decode(model.net, batch["tgt_in"], 0, kv, batch["src_mask"], None)
            sel = batch["tgt_out"] != -100
            h, targets = hidden[sel], batch["tgt_out"][sel]
        else:
            ids = batch["ids"]
            positions = torch.arange(ids.size(1), device=ids.device)[None].expand_as(ids)
            hidden, _ = decoder_forward(model.net, model.arch, ids, positions, None, None)
            labels = batch["labels"][:, 1:]
            sel = labels != -100
            h, targets = hidden[:, :-1][sel], labels[sel]
    n = targets.numel()
    return _chunked_ce(h, model.embedding, targets) / max(1, n), n

_BANNED = {}

def banned_ids(tokenizer, vocab):
    key = (id(tokenizer), vocab)
    if key not in _BANNED:
        ban = [tokenizer.token_to_id[t] for t in SPECIAL_TOKENS if t in tokenizer.token_to_id and t != "</s>"]
        ban += [tokenizer.token_to_id[str(b)] for b in list(range(0, 33)) + [127] if str(b) in tokenizer.token_to_id]
        ban = [i for i in ban if i < vocab]
        _BANNED[key] = torch.tensor(sorted(set(ban)), device=DEVICE, dtype=torch.long)
    return _BANNED[key]

@torch.no_grad()
def generate(model, tokenizer, prompts, min_new, max_new, decoding, batch_size, desc="generating"):
    """prompts: decoder models -> document ids followed by <s>; C -> document ids. Returns the generated summaries
    (text), in the order of the prompts. Greedy decoding with a repetition penalty and n-gram blocking; </s> is not
    allowed before min_new tokens; generation stops at </s> or max_new tokens."""
    model.eval()
    emb = model.embedding.float()
    vocab = emb.size(0)
    banned = banned_ids(tokenizer, vocab)
    eos, eow = tokenizer.eos_id, tokenizer.eow_id
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    out = [None] * len(prompts)
    rp, ng = float(decoding["repetition_penalty"]), int(decoding["no_repeat_ngram"])
    for bstart in tqdm(range(0, len(order), batch_size), desc=desc, leave=False):
        idx = order[bstart:bstart + batch_size]
        B = len(idx)
        P = max(len(prompts[i]) for i in idx)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            if model.arch == "C":
                src = torch.full((B, P), tokenizer.pad_id, dtype=torch.long, device=DEVICE)
                src_mask = torch.zeros((B, P), dtype=torch.bool, device=DEVICE)
                for r, i in enumerate(idx):
                    src[r, :len(prompts[i])] = torch.tensor(prompts[i], device=DEVICE)
                    src_mask[r, :len(prompts[i])] = True
                kv = c_encode(model.net, src, src_mask)
                cache = [None] * len(model.net.decoder)
                cur = torch.full((B, 1), tokenizer.bos_id, dtype=torch.long, device=DEVICE)
                hidden, cache = c_decode(model.net, cur, 0, kv, src_mask, cache)
                limit = min(max_new, model.net.max_tgt - 1)
            else:
                ids = torch.full((B, P), tokenizer.pad_id, dtype=torch.long, device=DEVICE)
                valid = torch.zeros((B, P), dtype=torch.bool, device=DEVICE)
                for r, i in enumerate(idx):
                    ids[r, P - len(prompts[i]):] = torch.tensor(prompts[i], device=DEVICE)
                    valid[r, P - len(prompts[i]):] = True
                pos = (valid.long().cumsum(1) - 1).clamp(min=0)
                ar = torch.arange(P, device=DEVICE)
                mask = (ar[None, :] <= ar[:, None])[None] & valid[:, None, :]
                mask = (mask | torch.eye(P, dtype=torch.bool, device=DEVICE)[None])[:, None]
                hidden, cache = decoder_forward(model.net, model.arch, ids, pos, mask, None)
                n_valid = valid.sum(1)
                limit = min(max_new, model.context - P)
        last = hidden[:, -1].float()
        seen = torch.zeros((B, vocab), dtype=torch.bool, device=DEVICE)
        gen = [[] for _ in range(B)]
        grams = [dict() for _ in range(B)]
        done = torch.zeros(B, dtype=torch.bool, device=DEVICE)
        prev_tok = torch.full((B,), -1, dtype=torch.long, device=DEVICE)
        for step in range(max(0, limit)):
            scores = F.linear(last, emb)
            scores[:, banned] = float("-inf")
            if step < min_new:
                scores[:, eos] = float("-inf")
            scores[prev_tok == eow, eow] = float("-inf")
            if rp != 1.0 and step > 0:
                s = scores[seen]
                scores[seen] = torch.where(s > 0, s / rp, s * rp)
            if ng > 1 and step >= ng - 1:
                for r in range(B):
                    blocked = grams[r].get(tuple(gen[r][-(ng - 1):]))
                    if blocked:
                        trial = scores[r].clone()
                        trial[list(blocked)] = float("-inf")
                        if torch.isfinite(trial).any():
                            scores[r] = trial
            nxt = scores.argmax(1)
            nxt = torch.where(done, torch.full_like(nxt, eos), nxt)
            nl = nxt.tolist()
            for r in range(B):
                if not bool(done[r]):
                    if ng > 1 and len(gen[r]) >= ng - 1:
                        grams[r].setdefault(tuple(gen[r][-(ng - 1):]), set()).add(nl[r])
                    gen[r].append(nl[r])
            seen[torch.arange(B, device=DEVICE), nxt] |= nxt != eow
            done |= nxt == eos
            prev_tok = nxt
            if bool(done.all()):
                break
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                if model.arch == "C":
                    hidden, cache = c_decode(model.net, nxt[:, None], step + 1, kv, src_mask, cache)
                else:
                    valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=DEVICE)], 1)
                    hidden, cache = decoder_forward(model.net, model.arch, nxt[:, None], n_valid[:, None],
                                                    valid[:, None, None, :], cache)
                    n_valid = n_valid + 1
            last = hidden[:, -1].float()
        for r, i in enumerate(idx):
            toks = [t for t in gen[r] if t != eos]
            out[i] = tokenizer.decode(toks)
    return out

## 9. Training

Batches hold as many (document, summary) sequences as fit in 8,192 tokens. One held-out evaluation per epoch; the best epoch is kept, and an interrupted run resumes after its last finished epoch.

In [9]:
TOKENS = {}
MAX_DOC_TOKENS = 60_000

def tokenized(slot):
    """{split: {"doc": [np.int32 ids], "summ": [np.int32 ids] (train/val)}} for the tokenizer of model `slot`."""
    tokenizer_dir = PHASE1[slot]["tokenizer_dir"]
    sizes = "-".join(str(len(CASES[s])) for s in ("train", "val", "test"))
    key = hashlib.md5(f"{tokenizer_dir}|{sizes}|{MAX_DOC_TOKENS}|{SEED}".encode()).hexdigest()[:12]
    if key in TOKENS:
        return TOKENS[key]
    path = os.path.join(CACHE_DIR, f"tokens_{key}.pkl")
    if os.path.exists(path):
        with open(path, "rb") as f:
            TOKENS[key] = pickle.load(f)
        print(f"tokenised data for model {slot}: loaded from {path}")
        return TOKENS[key]
    tok = tokenizer_for(slot)
    out = {}
    for split in ("train", "val", "test"):
        out[split] = {"doc": [np.asarray(tok.encode(c["document"], MAX_DOC_TOKENS), dtype=np.int32)
                              for c in tqdm(CASES[split], desc=f"tokenising {split} documents for model {slot}")]}
        if split != "test":
            out[split]["summ"] = [np.asarray(tok.encode(c["summary"]), dtype=np.int32) for c in CASES[split]]
    with open(path + ".tmp", "wb") as f:
        pickle.dump(out, f, protocol=4)
    os.replace(path + ".tmp", path)
    TOKENS[key] = out
    return out

def budgets(model, cfg):
    """Token budgets from the data: summary = 90th percentile of the reference length (at most half of a decoder's
    context, or C's decoder length); document = the rest of the context (C: its encoder length).
    Generation: at least half the median reference length, at most the summary budget."""
    data = tokenized(cfg["backbone"])
    lengths = np.array([len(s) for s in data["train"]["summ"]])
    p90, median = int(np.percentile(lengths, 90)), int(np.median(lengths))
    if model.arch == "C":
        summ = max(16, min(p90, model.net.max_tgt - 1))
        doc = model.net.max_src
    else:
        summ = max(16, min(p90, model.context // 2))
        doc = model.context - summ - 2
    return {"doc": int(doc), "summ": int(summ), "min_new": int(min(summ, max(8, median // 2))), "max_new": int(summ)}

def examples(cfg, b, split, extra=None):
    """-> list of (document ids, summary ids) within the budgets; `extra` = more (doc index, summary ids) pairs."""
    data = tokenized(cfg["backbone"])[split]
    pairs = [(select_document_tokens(d, b["doc"], cfg["doc_selection"]), list(s[:b["summ"]]))
             for d, s in zip(data["doc"], data["summ"])]
    for i, s in (extra or []):
        pairs.append((select_document_tokens(data["doc"][i], b["doc"], cfg["doc_selection"]), list(s[:b["summ"]])))
    return pairs

def prompts_for(model, cfg, b, split, indices=None):
    tok = tokenizer_for(cfg["backbone"])
    docs = tokenized(cfg["backbone"])[split]["doc"]
    indices = range(len(docs)) if indices is None else indices
    out = []
    for i in indices:
        d = select_document_tokens(docs[i], b["doc"], cfg["doc_selection"])
        out.append(d if model.arch == "C" else d + [tok.bos_id])
    return out

def token_batches(pairs, tokens_per_batch, order):
    """Group examples (in the given order) so that batch size x longest sequence <= tokens_per_batch."""
    out, cur, longest = [], [], 0
    for i in order:
        n = len(pairs[i][0]) + len(pairs[i][1]) + 2
        if cur and max(longest, n) * (len(cur) + 1) > tokens_per_batch:
            out.append(cur)
            cur, longest = [], 0
        cur.append(i)
        longest = max(longest, n)
    if cur:
        out.append(cur)
    return out

def collate(model, tok, pairs, idx):
    if model.arch == "C":
        S = max(len(pairs[i][0]) for i in idx) or 1
        T = max(len(pairs[i][1]) for i in idx) + 1
        src = np.full((len(idx), S), tok.pad_id, dtype=np.int64)
        src_mask = np.zeros((len(idx), S), dtype=bool)
        tin = np.full((len(idx), T), tok.pad_id, dtype=np.int64)
        tout = np.full((len(idx), T), -100, dtype=np.int64)
        for r, i in enumerate(idx):
            d, s = pairs[i]
            if not d:
                d = [tok.eow_id]
            src[r, :len(d)], src_mask[r, :len(d)] = d, True
            tin[r, :len(s) + 1] = [tok.bos_id] + s
            tout[r, :len(s) + 1] = s + [tok.eos_id]
        t = lambda a: torch.from_numpy(a).to(DEVICE)
        return {"src": t(src), "src_mask": t(src_mask), "tgt_in": t(tin), "tgt_out": t(tout)}
    L = max(len(pairs[i][0]) + len(pairs[i][1]) + 2 for i in idx)
    ids = np.full((len(idx), L), tok.pad_id, dtype=np.int64)
    labels = np.full((len(idx), L), -100, dtype=np.int64)
    for r, i in enumerate(idx):
        d, s = pairs[i]
        seq = list(d) + [tok.bos_id] + list(s) + [tok.eos_id]
        ids[r, :len(seq)] = seq
        labels[r, len(d) + 1:len(seq)] = list(s) + [tok.eos_id]
    return {"ids": torch.from_numpy(ids).to(DEVICE), "labels": torch.from_numpy(labels).to(DEVICE)}

@torch.no_grad()
def held_out_loss(model, tok, pairs, tokens_per_batch):
    model.eval()
    total, n = 0.0, 0
    for idx in token_batches(pairs, tokens_per_batch, sorted(range(len(pairs)), key=lambda i: len(pairs[i][0]) + len(pairs[i][1]))):
        loss, k = training_loss(model, collate(model, tok, pairs, idx))
        total += float(loss) * k
        n += k
    return total / max(1, n)

def exp_dir(name):
    path = os.path.join(WORK_ROOT, name)
    os.makedirs(path, exist_ok=True)
    return path

def build_model(cfg, verbose=True):
    return SummModel(cfg, verbose=verbose).to(DEVICE)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_percent": 100.0 * trainable / max(1, total)}

def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}

def load_trainable(model, state):
    result = model.load_state_dict(state, strict=False)
    if result.unexpected_keys:
        raise RuntimeError("Saved weights do not fit this model: " + str(result.unexpected_keys[:5]))

def save_atomic(obj, path):
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def config_signature(cfg, extra=""):
    entry = PHASE1[cfg["backbone"]]
    payload = {"cfg": {k: v for k, v in cfg.items() if k not in ("name", "gen_batch")},
               "phase1": [entry["config"], entry["eval_loss"]],
               "data": [len(CASES[s]) for s in ("train", "val", "test")], "extra": extra}
    return hashlib.md5(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()

def optimizer_groups(model, cfg):
    peft_lr = cfg["lr_peft"] if cfg["technique"] in ("bitfit", "adapter", "pfeiffer") else cfg["lr_backbone"]
    buckets = defaultdict(list)
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        lr = cfg["lr_peft"] if any(m in name for m in NEW_WEIGHT_MARKERS) else peft_lr
        decay = cfg["weight_decay"] if p.ndim >= 2 else 0.0
        buckets[(lr, decay)].append(p)
    return [{"params": params, "lr": lr, "base_lr": lr, "weight_decay": decay} for (lr, decay), params in buckets.items()]

def lr_factor(step, total_steps, warmup_frac):
    warmup = max(1, int(total_steps * warmup_frac))
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

def release(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def train_model(cfg, signature, extra_pairs=None):
    """Teacher-forced training on the reference summaries (+ teacher summaries for KD). One held-out evaluation per
    epoch (loss on the held-out reference summaries); the best epoch is kept. Resumes after the last finished epoch."""
    folder = exp_dir(cfg["name"])
    last_path, model_path = os.path.join(folder, "last.pt"), os.path.join(folder, "model.pt")
    torch.manual_seed(cfg["seed"])
    model = build_model(cfg)
    tok = tokenizer_for(cfg["backbone"])
    b = budgets(model, cfg)
    params = count_parameters(model)
    print(f"[{cfg['name']}] parameters: {params['total'] / 1e6:.2f}M total, {params['trainable'] / 1e6:.3f}M trained "
          f"({params['trainable_percent']:.2f}%) | budgets: document {b['doc']} tokens, summary {b['summ']} tokens "
          f"(generation {b['min_new']}-{b['max_new']})")
    train_pairs = examples(cfg, b, "train", extra_pairs)
    val_pairs = examples(cfg, b, "val")
    epoch_batches = [token_batches(train_pairs, cfg["tokens_per_batch"],
                                   np.random.RandomState(cfg["seed"] * 1000 + e).permutation(len(train_pairs)))
                     for e in range(cfg["epochs"])]
    total_steps = sum(len(x) for x in epoch_batches)
    epoch_tokens = sum(len(d) + len(t) + 2 for d, t in train_pairs)
    print(f"[{cfg['name']}] {len(train_pairs):,} training examples, {epoch_tokens / 1e6:.1f}M tokens per epoch, "
          f"{len(epoch_batches[0]):,} batches per epoch")
    optimizer = torch.optim.AdamW(optimizer_groups(model, cfg), betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    state = {"epoch": 0, "step": 0, "history": [], "best_loss": float("inf"), "best_epoch": 0, "bad_epochs": 0,
             "seconds": 0.0, "peak_memory_mb": 0.0, "skipped": 0}
    if os.path.exists(last_path):
        saved = torch_load(last_path)
        if saved.get("signature") == signature:
            load_trainable(model, saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            scaler.load_state_dict(saved["scaler"])
            state = saved["state"]
            print(f"[{cfg['name']}] resuming after epoch {state['epoch']}")
        del saved
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for epoch in range(state["epoch"], cfg["epochs"]):
        if state["bad_epochs"] >= cfg["patience"]:
            print(f"[{cfg['name']}] early stop: no improvement for {cfg['patience']} epochs.")
            break
        model.train()
        torch.manual_seed(cfg["seed"] * 1000 + epoch)
        t0, loss_sum, n_loss = time.time(), 0.0, 0
        bar = tqdm(epoch_batches[epoch], desc=f"{cfg['name']} epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for idx in bar:
            for g in optimizer.param_groups:
                g["lr"] = g["base_lr"] * lr_factor(state["step"], total_steps, cfg["warmup_frac"])
            loss, _ = training_loss(model, collate(model, tok, train_pairs, idx))
            optimizer.zero_grad(set_to_none=True)
            state["step"] += 1
            if not torch.isfinite(loss):
                state["skipped"] += 1
                if state["skipped"] in (1, 10, 100) or state["skipped"] % 1000 == 0:
                    print(f"[{cfg['name']}] WARNING: non-finite loss, batch skipped ({state['skipped']} so far)", flush=True)
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item()
            n_loss += 1
            if n_loss % 20 == 0:
                bar.set_postfix(loss=f"{loss_sum / n_loss:.4f}")
        bar.close()
        train_seconds = time.time() - t0
        val_loss = held_out_loss(model, tok, val_pairs, cfg["tokens_per_batch"])
        record = {"epoch": epoch + 1, "train_loss": loss_sum / max(1, n_loss), "val_loss": val_loss, "seconds": train_seconds}
        message = (f"[{cfg['name']}] epoch {epoch + 1}/{cfg['epochs']} | train loss {record['train_loss']:.4f} | "
                   f"held-out loss {val_loss:.4f} (perplexity {math.exp(min(20, val_loss)):.1f})")
        if val_loss < state["best_loss"]:
            state.update(best_loss=val_loss, best_epoch=epoch + 1, bad_epochs=0)
            save_atomic({"model": trainable_state(model), "signature": signature, "epoch": epoch + 1}, model_path)
            message += "  <- best"
        else:
            state["bad_epochs"] += 1
        state["history"].append(record)
        state["epoch"] = epoch + 1
        state["seconds"] += train_seconds
        if DEVICE.type == "cuda":
            state["peak_memory_mb"] = max(state["peak_memory_mb"], torch.cuda.max_memory_allocated() / 1e6)
        save_atomic({"model": trainable_state(model), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                     "state": state, "signature": signature}, last_path)
        print(message + f" | {train_seconds / 60:.1f} min", flush=True)
    load_trainable(model, torch_load(model_path)["model"])
    if os.path.exists(last_path):
        os.remove(last_path)
    info = {"best_epoch": state["best_epoch"], "best_val_loss": state["best_loss"], "history": state["history"],
            "train_seconds": state["seconds"], "peak_memory_mb": state["peak_memory_mb"], "parameters": params,
            "budgets": b, "train_examples": len(train_pairs), "skipped_batches": state["skipped"]}
    return model, info

def load_model(cfg):
    model = build_model(cfg, verbose=False)
    load_trainable(model, torch_load(os.path.join(exp_dir(cfg["name"]), "model.pt"))["model"])
    return model

## 10. Run the experiments

For each experiment: train → summarise held-out documents with each decoding setting → test summaries with the best one → descriptive zip in `submissions/`. Everything finished is loaded from disk. One failing experiment does not stop the others.

In [10]:
RESULTS = {}

def write_predictions(cfg, folder, summaries, val_rl):
    """prediction.json = [{"id": <document id exactly as in test.json>, "summary": "..."}] (the list format), one
    non-empty summary per test document, zipped under a descriptive name in the experiment folder AND submissions/."""
    output = [{"id": c["id"], PREDICTION_KEY: s} for c, s in zip(CASES["test"], summaries)]
    assert len(output) == len(CASES["test"])
    prediction_path = os.path.join(folder, "prediction.json")
    with open(prediction_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False)
    zip_name = submission_name(cfg, val_rl)
    for old in glob.glob(os.path.join(folder, "phase6_*.zip")) + \
            glob.glob(os.path.join(SUBMISSION_DIR, f"phase6_{cfg['name'].split('_')[0]}_*.zip")):
        os.remove(old)
    zip_path = os.path.join(folder, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path, "r") as z:
        assert z.namelist() == ["prediction.json"]
        check = json.loads(z.read("prediction.json").decode("utf-8"))
    assert [r["id"] for r in check] == [c["id"] for c in CASES["test"]], "test ids changed"
    assert all(isinstance(r[PREDICTION_KEY], str) and r[PREDICTION_KEY].strip() for r in check), "empty summary"
    shutil.copy2(zip_path, os.path.join(SUBMISSION_DIR, zip_name))
    return os.path.join(SUBMISSION_DIR, zip_name)

def with_fallback(summaries, split):
    """A summary that came out empty is replaced by the first 60 words of its document (keeps the file valid)."""
    out, n = [], 0
    for s, c in zip(summaries, CASES[split]):
        if s and s.strip():
            out.append(s.strip())
        else:
            out.append(" ".join(c["document"].split()[:60]) or "no summary")
            n += 1
    if n:
        print(f"  {n} empty summaries replaced by the start of their document")
    return out

def cached_generation(path, key, fn):
    """Generated texts are saved with a key (model signature + settings) and re-used on re-runs."""
    if os.path.exists(path):
        saved = load_json_file(path)
        if saved.get("key") == key:
            return saved["texts"]
    texts = fn()
    write_json({"key": key, "texts": texts}, path)
    return texts

def teacher_pairs(cfg):
    """Sequence-level knowledge distillation (Kim & Rush): the teacher summarises kd_docs training documents; the
    student learns these summaries in addition to the reference summaries. The teacher's summaries are tokenised
    with the STUDENT's tokenizer, so teacher and student may use different vocabularies."""
    teacher_cfg = experiment_config(cfg["teacher"])
    folder = exp_dir(cfg["name"])
    n = min(cfg["kd_docs"], len(CASES["train"]))
    chosen = sorted(np.random.RandomState(cfg["seed"]).permutation(len(CASES["train"]))[:n].tolist())
    teacher_result = RESULTS[cfg["teacher"]]
    path = os.path.join(folder, "teacher_summaries.json")
    key = teacher_result["signature"] + f"|{n}|" + json.dumps(teacher_result["decoding"], sort_keys=True)

    def run():
        teacher = load_model(teacher_cfg)
        b = budgets(teacher, teacher_cfg)
        prompts = prompts_for(teacher, teacher_cfg, b, "train", chosen)
        texts = generate(teacher, tokenizer_for(teacher_cfg["backbone"]), prompts, b["min_new"], b["max_new"],
                         DECODING_CONFIGS[teacher_result["decoding"]], teacher_cfg["gen_batch"],
                         desc=f"teacher {cfg['teacher']} summaries")
        release(teacher)
        return texts

    texts = cached_generation(path, key, run)
    tok = tokenizer_for(cfg["backbone"])
    pairs = [(i, np.asarray(tok.encode(t), dtype=np.int32)) for i, t in zip(chosen, texts) if t.strip()]
    print(f"[{cfg['name']}] knowledge distillation: {len(pairs)} teacher summaries from {cfg['teacher']} "
          f"(mean {np.mean([len(t.split()) for t in texts]):.0f} words)")
    return pairs, key

def run_experiment(name):
    if name in RESULTS:
        return RESULTS[name]
    cfg = experiment_config(name)
    if cfg["backbone"] not in PHASE1:
        print(f"[{name}] SKIPPED: no Phase-1 checkpoint for model {cfg['backbone']} is attached.")
        return None
    extra_sig, extra_pairs = "", None
    if cfg["teacher"]:
        dep = run_experiment(cfg["teacher"])
        if dep is None:
            print(f"[{name}] SKIPPED: its teacher {cfg['teacher']} could not be run.")
            return None
        extra_pairs, extra_sig = teacher_pairs(cfg)
    folder = exp_dir(name)
    signature = config_signature(cfg, extra_sig)
    results_path = os.path.join(folder, "results.json")
    if os.path.exists(results_path):
        saved = load_json_file(results_path)
        if saved.get("signature") == signature and os.path.exists(saved.get("submission", "")):
            print(f"[{name}] already finished: held-out ROUGE-L {saved['val']['rougeL_f1']:.4f} (loaded from disk)")
            RESULTS[name] = saved
            return saved

    model_path, info_path = os.path.join(folder, "model.pt"), os.path.join(folder, "train_info.json")
    trained = (os.path.exists(model_path) and os.path.exists(info_path) and not os.path.exists(os.path.join(folder, "last.pt"))
               and torch_load(model_path).get("signature") == signature)
    if trained:
        model = load_model(cfg)
        info = load_json_file(info_path)
        print(f"[{name}] model already trained; loading it.")
    else:
        model, info = train_model(cfg, signature, extra_pairs)
        write_json(info, info_path)
    tok = tokenizer_for(cfg["backbone"])
    b = info["budgets"]

    n_eval = min(GEN_EVAL_DOCS, len(CASES["val"]))
    eval_idx = list(range(n_eval))
    refs = [CASES["val"][i]["summary"] for i in eval_idx]
    prompts = prompts_for(model, cfg, b, "val", eval_idx)
    val_scores = {}
    for dname, dcfg in DECODING_CONFIGS.items():
        t0 = time.time()
        texts = cached_generation(os.path.join(folder, f"val_{dname}.json"), signature + json.dumps(dcfg, sort_keys=True),
                                  lambda: generate(model, tok, prompts, b["min_new"], b["max_new"], dcfg, cfg["gen_batch"],
                                                   desc=f"{name} held-out ({dname})"))
        texts = with_fallback(texts, "val")
        sc = summary_scores(texts, refs)
        sc["bertscore_f1"] = bertscore_f1(texts, refs)
        sc["seconds"] = time.time() - t0
        val_scores[dname] = sc
        print(f"[{name}] held-out ({n_eval} documents), {dname}: ROUGE-1 {sc['rouge1_f1']:.4f} | ROUGE-L {sc['rougeL_f1']:.4f}"
              + (f" | BERTScore {sc['bertscore_f1']:.4f}" if sc["bertscore_f1"] is not None else "")
              + f" | {sc['words_mean']:.0f} words (references {sc['reference_words_mean']:.0f}) | repeated 4-grams "
              f"{100 * sc['repeated_4gram_share']:.1f}%", flush=True)
    key_metric = "bertscore_f1" if all(v["bertscore_f1"] is not None for v in val_scores.values()) else "rougeL_f1"
    best = max(val_scores, key=lambda d: val_scores[d][key_metric])
    print(f"[{name}] decoding for the test set: {best} (best {key_metric} on the held-out documents)")
    example = load_json_file(os.path.join(folder, f"val_{best}.json"))["texts"]

    t0 = time.time()
    test_prompts = prompts_for(model, cfg, b, "test")
    test_texts = cached_generation(os.path.join(folder, "test_summaries.json"),
                                   signature + json.dumps(DECODING_CONFIGS[best], sort_keys=True),
                                   lambda: generate(model, tok, test_prompts, b["min_new"], b["max_new"],
                                                    DECODING_CONFIGS[best], cfg["gen_batch"], desc=f"{name} test summaries"))
    test_texts = with_fallback(test_texts, "test")
    test_seconds = time.time() - t0
    zip_path = write_predictions(cfg, folder, test_texts, val_scores[best]["rougeL_f1"])
    result = {"name": name, "signature": signature, "config": {k: v for k, v in cfg.items() if k != "name"},
              "phase1_checkpoint": PHASE1[cfg["backbone"]]["dir"], "train": info, "decoding": best,
              "val": val_scores[best], "val_all_decodings": val_scores, "test_documents": len(test_texts),
              "test_words_mean": float(np.mean([len(t.split()) for t in test_texts])),
              "test_seconds": test_seconds, "submission": zip_path,
              "example": {"reference": refs[0][:1500], "generated": (example[0] if example else "")[:1500]}}
    write_json(result, results_path)
    print(f"[{name}] test: {len(test_texts):,} summaries, {result['test_words_mean']:.0f} words on average -> {zip_path}\n", flush=True)
    release(model)
    RESULTS[name] = result
    return result

for _name in RUN_EXPERIMENTS:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    try:
        run_experiment(_name)
    except Exception as _err:
        import traceback
        traceback.print_exc()
        print(f"[{_name}] FAILED: {_err} - continuing with the next experiment.\n")
        release()

Loaded Phase-1 model B (GQA/RoPE decoder): 8/8 layers, 66.07M parameters


tokenising train documents for model B:   0%|          | 0/6730 [00:00<?, ?it/s]

tokenising val documents for model B:   0%|          | 0/300 [00:00<?, ?it/s]

tokenising test documents for model B:   0%|          | 0/100 [00:00<?, ?it/s]

[S2_B_houlsby] parameters: 69.23M total, 3.173M trained (4.58%) | budgets: document 1022 tokens, summary 1024 tokens (generation 710-1024)
[S2_B_houlsby] 6,730 training examples, 13.2M tokens per epoch, 1,683 batches per epoch


S2_B_houlsby epoch 1/4:   0%|          | 0/1683 [00:00<?, ?it/s]

... (65 outputs left out)

S4_C_lora held-out (greedy_rep1.2_ng6):   0%|          | 0/4 [00:00<?, ?it/s]

[S4_C_lora] held-out (60 documents), greedy_rep1.2_ng6: ROUGE-1 0.3490 | ROUGE-L 0.2213 | 509 words (references 862) | repeated 4-grams 0.2%
[S4_C_lora] decoding for the test set: greedy_rep1.2_ng6 (best rougeL_f1 on the held-out documents)


S4_C_lora test summaries:   0%|          | 0/7 [00:00<?, ?it/s]

[S4_C_lora] test: 100 summaries, 509 words on average -> /kaggle/working/cs781_phase6/submissions/phase6_S4_C-EncDec_LoRA-all-r16_headtail_valRL0.2213.zip



## 11. Results and analysis

In [11]:
finished = {name: r for name, r in RESULTS.items() if r is not None}
for _name in EXPERIMENTS:
    _path = os.path.join(WORK_ROOT, _name, "results.json")
    if _name not in finished and os.path.exists(_path):
        finished[_name] = load_json_file(_path)

rows = []
for name, r in sorted(finished.items(), key=lambda kv: -kv[1]["val"]["rougeL_f1"]):
    cfg, v, t = r["config"], r["val"], r["train"]
    technique = cfg["technique"] + (f"-{cfg['peft_targets']}-r{cfg['lora_r']}" if cfg["technique"] in ("lora", "lorma") else "") \
        + (" + SeqKD" if cfg["teacher"] else "")
    rows.append({"experiment": name, "model": cfg["backbone"] + (f"[:{cfg['n_layers']}]" if cfg["n_layers"] else ""),
                 "technique": technique, "document": cfg["doc_selection"], "decoding": r["decoding"],
                 "val_rouge1": v["rouge1_f1"], "val_rougeL": v["rougeL_f1"],
                 "val_bertscore": v["bertscore_f1"] if v["bertscore_f1"] is not None else float("nan"),
                 "words": v["words_mean"], "ref_words": v["reference_words_mean"], "repeat4_%": 100 * v["repeated_4gram_share"],
                 "held_out_loss": t["best_val_loss"], "best_epoch": t["best_epoch"],
                 "params_M": t["parameters"]["total"] / 1e6, "trainable_M": t["parameters"]["trainable"] / 1e6,
                 "trainable_%": t["parameters"]["trainable_percent"], "train_min": t["train_seconds"] / 60,
                 "peak_mem_GB": t["peak_memory_mb"] / 1000, "test_min": r["test_seconds"] / 60,
                 "submission": os.path.basename(r["submission"])})

if not rows:
    print("No finished experiments yet.")
else:
    header = (f"{'experiment':18s} {'model':6s} {'technique':16s} {'R-1':>6s} {'R-L':>6s} {'BERTSc':>6s} {'words':>6s} {'ref':>5s} "
              f"{'rep4%':>5s} {'loss':>6s} {'ep':>3s} {'params':>7s} {'trained':>8s} {'%':>6s} {'train':>6s} {'mem':>5s} {'test':>6s}")
    print(f"Held-out documents ({min(GEN_EVAL_DOCS, len(CASES['val']))} from train.json, never trained on). R-1 / R-L = ROUGE F1 "
          f"(local); BERTSc = BERTScore F1 if the package is installed; rep4% = repeated word 4-grams; loss = held-out "
          f"cross-entropy on the reference summaries.\n" + header + "\n" + "-" * len(header))
    for x in rows:
        print(f"{x['experiment']:18s} {x['model']:6s} {x['technique']:16s} {x['val_rouge1']:6.4f} {x['val_rougeL']:6.4f} "
              f"{x['val_bertscore']:6.4f} {x['words']:6.0f} {x['ref_words']:5.0f} {x['repeat4_%']:5.1f} {x['held_out_loss']:6.3f} "
              f"{x['best_epoch']:3d} {x['params_M']:6.1f}M {x['trainable_M']:7.3f}M {x['trainable_%']:6.2f} "
              f"{x['train_min']:5.1f}m {x['peak_mem_GB']:4.1f}G {x['test_min']:5.1f}m")
    with open(os.path.join(WORK_ROOT, "results_summary.csv"), "w", encoding="utf-8") as f:
        f.write(",".join(rows[0].keys()) + "\n")
        for x in rows:
            f.write(",".join(str(round(v, 5)) if isinstance(v, float) else str(v) for v in x.values()) + "\n")
    print("\nDecoding settings compared (held-out ROUGE-L):")
    for x in rows:
        print(f"  {x['experiment']:18s} " + " | ".join(f"{d}: {s['rougeL_f1']:.4f}"
                                                       for d, s in finished[x["experiment"]]["val_all_decodings"].items()))
    ex = finished[rows[0]["experiment"]]["example"]
    print(f"\nExample ({rows[0]['experiment']}), first held-out document:\nREFERENCE: {ex['reference'][:700]}\n"
          f"GENERATED: {ex['generated'][:700]}")
    print(f"\nSummary table saved to {os.path.join(WORK_ROOT, 'results_summary.csv')}")

Held-out documents (60 from train.json, never trained on). R-1 / R-L = ROUGE F1 (local); BERTSc = BERTScore F1 if the package is installed; rep4% = repeated word 4-grams; loss = held-out cross-entropy on the reference summaries.
experiment         model  technique           R-1    R-L BERTSc  words   ref rep4%   loss  ep  params  trained      %  train   mem   test
-----------------------------------------------------------------------------------------------------------------------------------------
S4_C_lora          C      lora-all-r16     0.3490 0.2213    nan    509   862   0.2  2.856   4  108.8M   3.244M   2.98  39.8m  2.8G   2.2m
S2_B_houlsby       B      adapter          0.3980 0.2127    nan    405   862   4.8  1.432   4   69.2M   3.173M   4.58  43.7m  3.3G   2.0m
S1_D_lorma         D      lorma-qv-r32     0.3895 0.2109    nan    423   862   6.6  1.489   4   65.4M   0.786M   1.20  40.0m  2.6G   3.1m
S3_A4_kd           A[:4]  full + SeqKD     0.3741 0.2064    nan    385   862   4.

## 12. Submission files

In [12]:
if rows:
    print(f"Submission files in {SUBMISSION_DIR} (best held-out ROUGE-L first):")
    for x in rows:
        print(f"  ROUGE-L {x['val_rougeL']:.4f}   {x['submission']}")
    with zipfile.ZipFile(os.path.join(SUBMISSION_DIR, rows[0]["submission"])) as z:
        _p = json.loads(z.read("prediction.json"))
    print(f"\nprediction.json: {len(_p)} summaries; first one: {{'id': {_p[0]['id']!r}, 'summary': {_p[0]['summary'][:200]!r}...}}")

Submission files in /kaggle/working/cs781_phase6/submissions (best held-out ROUGE-L first):
  ROUGE-L 0.2213   phase6_S4_C-EncDec_LoRA-all-r16_headtail_valRL0.2213.zip
  ROUGE-L 0.2127   phase6_S2_B-GQA-RoPE_HoulsbyAdapter_headtail_valRL0.2127.zip
  ROUGE-L 0.2109   phase6_S1_D-GPT400M_LoRMA-qv-r32_headtail_valRL0.2109.zip
  ROUGE-L 0.2064   phase6_S3_A-GPT220M-first4layers_SeqKD-from-S2_headtail_valRL0.2064.zip

prediction.json: 100 summaries; first one: {'id': '4963', 'summary': 'The appellant was a cufact of the respondent and the appellant in the respondent in the Pe of the State of the Par and the State Government to the State in the State and the order of the High Court un'...}
